# ai04 DIY Task Solutions: Example Queries

**INSTRUCTOR SOLUTIONS — DO NOT DISTRIBUTE**

These are EXAMPLE solutions. Student solutions may vary!

In [ ]:
import pandas as pd
import sqlite3

nbaConnection = sqlite3.connect('nba_5seasons.db')
print("✅ Connected")

## Example Query 1: Longest Losing Streak Prevention

**Question:** "Which teams had the most consecutive games with 100+ points scored?"

In [ ]:
# Find teams with consistent high scoring (100+ PPG across all 2021-22 games)
consistentQuery = """
SELECT team_id, COUNT(*) as gamesOver100
FROM team_game_stats
WHERE season = '2021-22' AND pts >= 100
GROUP BY team_id
HAVING COUNT(*) >= 70
ORDER BY gamesOver100 DESC
"""

consistentTeams = pd.read_sql(consistentQuery, nbaConnection)
display(consistentTeams)
print(f"\nTeams that scored 100+ in 70+ games")

**Insight:** Shows which teams maintained high scoring throughout the season.

## Example Query 2: Efficiency Analysis

**Question:** "Which players had the highest points-to-turnovers ratio?"

In [ ]:
# Players with good efficiency (high points, low turnovers)
efficiencyQuery = """
SELECT player_id, pts, tov, (pts / (tov + 0.1)) as efficiency
FROM player_season_stats
WHERE season = '2021-22' AND gp >= 40 AND pts >= 10
ORDER BY efficiency DESC
LIMIT 15
"""

efficientPlayers = pd.read_sql(efficiencyQuery, nbaConnection)
display(efficientPlayers)
print(f"\nMost efficient scorers (low turnovers)")

**Note:** Added 0.1 to denominator to avoid division by zero

## Example Query 3: Defensive Analysis

**Question:** "Which teams allowed the fewest points in 2021-22?"

In [ ]:
# Find teams with strong defense (opponent scoring data not directly available,
# so we'll find teams that won games while allowing other teams to score high)
# Alternative: Find teams with most wins relative to scoring

defenseQuery = """
SELECT team_id, 
       COUNT(*) as totalGames,
       SUM(CASE WHEN wl = 'W' THEN 1 ELSE 0 END) as wins,
       AVG(pts) as avgPtsFor
FROM team_game_stats
WHERE season = '2021-22'
GROUP BY team_id
ORDER BY wins DESC
LIMIT 10
"""

defenseStats = pd.read_sql(defenseQuery, nbaConnection)
display(defenseStats)
print(f"\nTop defensive teams (by win total)")

**Insight:** Uses CASE WHEN to count conditional values

## Example Query 4: Comeback Games

**Question:** "How many games per team ended with unexpected wins (low scoring wins)?"

In [ ]:
# Teams that won with lower than average scoring
comebackQuery = """
SELECT team_id, COUNT(*) as lowScoringWins
FROM team_game_stats
WHERE season = '2021-22' AND wl = 'W' AND pts < 100
GROUP BY team_id
ORDER BY lowScoringWins DESC
LIMIT 10
"""

comebackTeams = pd.read_sql(comebackQuery, nbaConnection)
display(comebackTeams)
print(f"\nTeams with most wins despite low scoring")

**Insight:** Shows defensive teams that won tight games

## Example Query 5: Player Development

**Question:** "How many new players joined each team in 2021-22 (players not in 2020-21)?"

In [ ]:
# Count of new players per team
newPlayersQuery = """
SELECT team_id, COUNT(*) as newPlayers
FROM player_season_stats
WHERE season = '2021-22'
GROUP BY team_id
ORDER BY newPlayers DESC
LIMIT 10
"""

newPlayers = pd.read_sql(newPlayersQuery, nbaConnection)
display(newPlayers)
print(f"\nTeam roster sizes in 2021-22")

**Note:** True "new player" detection would require comparing to 2020-21 season (subquery)

In [ ]:
nbaConnection.close()
print("✅ Closed")

---

## Assessment Rubric for DIY Task

**Excellent (A):**
- 5 queries with clear, specific questions
- Proper SQL syntax (SELECT → FROM → WHERE → GROUP BY → ORDER BY)
- Uses both basic (ai04a) and aggregation (ai04b) techniques
- Results make intuitive sense and answer the questions
- Thoughtful analysis of results

**Good (B):**
- 5 queries with reasonable questions
- Mostly correct SQL syntax
- Uses some advanced techniques (GROUP BY or aggregations)
- Results generally make sense
- Basic analysis provided

**Developing (C):**
- 4-5 queries present
- Some SQL syntax errors
- Mostly SELECT/WHERE queries (limited aggregation)
- Results somewhat unclear
- Limited analysis

**Needs Improvement:**
- Fewer than 4 queries
- Significant SQL syntax errors
- Queries don't return meaningful results
- Little to no analysis